# 04.2 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/02-data-leakage/exercises.ipynb)

Run the setup cell first.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

# the column the old export "helpfully" added: days since last order, measured on the EXPORT date
export_date = sales["InvoiceDate"].max()
last_ever = sales.groupby("Customer ID")["InvoiceDate"].max()
cust["recency_days"] = (export_date - last_ever).dt.days.reindex(cust.index)
print("export date:", export_date.date())
cust[["days_since_last", "recency_days", "churned"]].head()

3,317 customers, churn rate 43.6%
export date: 2011-12-09


,days_since_last,recency_days,churned
Customer ID,,,
12346.0,225,325,1
12347.0,29,1,0
12348.0,148,74,0
12350.0,210,309,1
12352.0,162,35,0


## Exercise 1: when is each feature known?

For each candidate feature, say whether it may be used to predict churn at the 1 September cut-off: (a) number of orders before 1 Sep, (b) total spend in 2011, (c) whether the customer opened the October newsletter, (d) country, (e) number of support tickets ever filed (from a table exported in December), (f) days since first order, measured at 1 Sep. Then add feature (b) to the churn table and see what it does to a depth-4 tree.

In [2]:
# your code here

## Exercise 2: find the leak with one line

Without being told which column leaks, train a depth-4 tree on `features + ["recency_days"]` and print the feature importances. Then compute the correlation of every feature with `churned`. Which check would have caught the leak fastest?

In [3]:
# your code here

## Exercise 3: contamination grows with the number of columns

Repeat the noise experiment (select the 20 most correlated columns using all rows, then split) for 50, 200, 1,000 and 5,000 noise columns. Plot the fake accuracy against the number of columns.

In [4]:
# your code here

## Exercise 4: duplicates and model flexibility

On the duplicated table (30% of rows repeated), compare a depth-3 tree, an unlimited tree and a 1-nearest-neighbour model. Which one gains most from the duplicates, and why?

In [5]:
# your code here